# Task 1 — Data Cleaning & Preprocessing
**AI & ML Internship | Titanic dataset**

This notebook follows the assignment steps: explore the data, handle missing values, encode categories, standardize numeric features, visualize outliers with boxplots, and remove selected outliers.

Dataset source specified by the assignment: [Yasser H. Titanic dataset](https://www.kaggle.com/datasets/yasserh/titanic-dataset). The CSV is included in `dataset/Titanic-Dataset.csv`.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
DATA = ROOT / 'dataset' / 'Titanic-Dataset.csv'
OUT = ROOT / 'outputs'
OUT.mkdir(exist_ok=True)
sns.set_theme(style='whitegrid')
df = pd.read_csv(DATA)
print('Dataset shape:', df.shape)
display(df.head())


## 1. Initial exploration
Inspect dimensions, data types, descriptive statistics, and missing values before cleaning.


In [ ]:
print('Rows and columns:', df.shape)
df.info()
display(df.describe(include='all').T)
missing_before = df.isna().sum().sort_values(ascending=False)
display(missing_before.to_frame('missing_count'))
plt.figure(figsize=(10,5))
sns.barplot(x=missing_before.index, y=missing_before.values)
plt.xticks(rotation=45, ha='right')
plt.title('Missing values before cleaning')
plt.tight_layout()
plt.savefig(OUT / 'missing_values_before.png', dpi=160)
plt.show()


## 2. Handle missing values

- Drop `Cabin` because it is very sparse.
- Fill missing `Age` using the median within each passenger class, then overall median.
- Fill `Fare` with its median and `Embarked` with its mode.
- Remove rows without the target `Survived`, if present.


In [ ]:
data = df.copy()
dropped_columns = []
if 'Cabin' in data:
    data = data.drop(columns='Cabin')
    dropped_columns.append('Cabin')
age_medians = data.groupby('Pclass', observed=False)['Age'].transform('median')
data['Age'] = data['Age'].fillna(age_medians).fillna(data['Age'].median())
data['Fare'] = data['Fare'].fillna(data['Fare'].median())
if data['Embarked'].isna().any():
    data['Embarked'] = data['Embarked'].fillna(data['Embarked'].mode().iloc[0])
data = data.dropna(subset=['Survived']).copy()
display(data.isna().sum().to_frame('remaining_missing_values'))


## 3. Detect and remove outliers

Use boxplots and the 1.5 × IQR rule on `Age` and `Fare`. Extreme values are not automatically errors; removal here demonstrates the assignment requirement.


In [ ]:
outlier_columns = ['Age', 'Fare']
fig, axes = plt.subplots(1, 2, figsize=(11,4))
for ax, col in zip(axes, outlier_columns):
    sns.boxplot(y=data[col], ax=ax)
    ax.set_title(f'{col} — before IQR filtering')
plt.tight_layout()
plt.savefig(OUT / 'boxplots_before_outlier_removal.png', dpi=160)
plt.show()

keep = pd.Series(True, index=data.index)
outlier_bounds = {}
for col in outlier_columns:
    q1, q3 = data[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    outlier_bounds[col] = (float(lower), float(upper))
    keep &= data[col].between(lower, upper)
rows_before = len(data)
data = data.loc[keep].copy()
print('Rows before:', rows_before, '| after:', len(data), '| removed:', rows_before-len(data))
print('IQR bounds:', outlier_bounds)

fig, axes = plt.subplots(1, 2, figsize=(11,4))
for ax, col in zip(axes, outlier_columns):
    sns.boxplot(y=data[col], ax=ax)
    ax.set_title(f'{col} — after IQR filtering')
plt.tight_layout()
plt.savefig(OUT / 'boxplots_after_outlier_removal.png', dpi=160)
plt.show()


## 4. Encode categorical variables

Use one-hot encoding for nominal `Sex` and `Embarked` categories. Drop identifier/free-text fields that are not part of this basic feature table.


In [ ]:
for col in ['PassengerId', 'Name', 'Ticket']:
    if col in data:
        data = data.drop(columns=col)
        dropped_columns.append(col)
data = pd.get_dummies(data, columns=['Sex', 'Embarked'], drop_first=True, dtype=int)
display(data.head())
print('Columns:', data.columns.tolist())


## 5. Standardize numerical features

`StandardScaler` transforms `Age` and `Fare` to zero mean and unit variance. The target `Survived` is not scaled.


In [ ]:
scale_columns = ['Age', 'Fare']
scaler = StandardScaler()
data[scale_columns] = scaler.fit_transform(data[scale_columns])
display(data[scale_columns].agg(['mean','std','min','max']).round(3))
plt.figure(figsize=(8,4))
sns.histplot(data['Age'], bins=25, kde=True)
plt.title('Standardized age distribution')
plt.tight_layout()
plt.savefig(OUT / 'age_distribution.png', dpi=160)
plt.show()


## 6. Save processed data and summary


In [ ]:
output_csv = OUT / 'titanic_cleaned_encoded_scaled.csv'
data.to_csv(output_csv, index=False)
summary = {
    'original_rows': int(df.shape[0]), 'original_columns': int(df.shape[1]),
    'final_rows': int(data.shape[0]), 'final_columns': int(data.shape[1]),
    'rows_removed_as_outliers': int(rows_before-len(data)),
    'dropped_columns': dropped_columns, 'scaled_columns': scale_columns,
    'remaining_missing_values': int(data.isna().sum().sum()),
    'outlier_bounds': outlier_bounds, 'output_csv': str(output_csv)
}
summary_text = '\n'.join(f'{k}: {v}' for k,v in summary.items())
(OUT / 'preprocessing_summary.txt').write_text(summary_text, encoding='utf-8')
print(summary_text)
display(data.head())


## 7. Conclusion

The dataset was explored, missing values were imputed, sparse/unneeded columns were dropped, categorical features were one-hot encoded, numerical features were standardized, and IQR outliers were removed after boxplot inspection.

**Methodological caution:** In a predictive ML workflow, split the data first and fit imputers, encoders, scalers, and outlier thresholds on training data only (preferably with a scikit-learn `Pipeline`) to avoid data leakage. Validate whether unusual fares/ages are legitimate before removing them.
